## Folium

In [ ]:
import os
import matplotlib.pyplot as plt
import koreanize_matplotlib

import numpy as np
import pandas as pd
import geopandas as gpd

from urllib.request import urlopen
import json


import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import folium



### 1. Gap Minder Data 불러오기

In [ ]:
# Gap Minder Data 불러오기
# 대륙, 나라, 연도, 수명, 인구수, GDP ...
gapminder = px.data.gapminder()
gapminder

In [ ]:
# year 2007만 사용
gapminder_2007 = gapminder.query("year == 2007") # query()라는 함수를 사용하여 2007 추출
gapminder_2007

## GeoJSON & GeoPandas
- **GeoJSON** : 지리 정보(좌표, 도형)를 표현하는 JSON 기반의 공개 표준 포맷
  - `FeatureCollection` 안에 여러 개의 `Feature`가 들어있는 구조
  - 각 `Feature`는 `properties`(속성 정보)와 `geometry`(좌표, `Point`/`LineString`/`Polygon`/`MultiPolygon` 등)로 구성
  - 공식 사이트: https://geojson.org/
- **GeoPandas** : pandas의 DataFrame에 공간 정보(`geometry`)를 더한 `GeoDataFrame`을 다루는 라이브러리
  - `gpd.read_file()`로 GeoJSON/Shapefile 등을 바로 불러와 지도 데이터를 표 형태로 다룰 수 있음
  - `geometry` 컬럼은 Shapely 객체 → 면적, 거리, 중첩(intersects) 등 공간 연산과 `.plot()` 시각화 지원
  - 공식 사이트: https://geopandas.org/

- pip install geopandas

### 2. GeoPandas로 world-countries.geojson 불러오기
- `gpd.read_file()`로 GeoJSON을 바로 `GeoDataFrame`으로 읽음
- `geometry` 컬럼에 Shapely 도형(Polygon/MultiPolygon)이 들어있어 pandas처럼 다루면서 공간 연산도 가능

In [ ]:
import geopandas as gpd

world = gpd.read_file("data/world-countries.geojson")

print(world.shape)
print(world.crs)             # 좌표계 확인 (EPSG:4326, 위경도)
world.head()

In [ ]:
# 도형 타입 분포 (섬이 있는 나라는 MultiPolygon)
world.geom_type.value_counts()

In [ ]:
world.plot(figsize=(12, 8), color="white", edgecolor="black")
plt.savefig('image/6_1.png')

In [ ]:
# id(ISO-3 코드)로 특정 국가 찾아서 지도로 확인
world[world["id"] == "KOR"].plot()
plt.savefig('image/6_2.png')

### 3. world ↔ gapminder_2007 조인
- `world`(GeoDataFrame)의 `id`(ISO-3 코드)와 `gapminder_2007`의 `iso_alpha`를 조인 키로 사용
- `world`를 왼쪽에 두고 `left` 조인 → `world`의 177개 국가는 모두 유지되고, 매칭되는 gapminder 통계만 붙음(매칭 안 되면 결측)

In [ ]:
world_gapminder = world.merge(
    gapminder_2007,
    left_on="id",
    right_on="iso_alpha",
    how="left",
)

print(type(world_gapminder))
print(len(world), "->", len(world_gapminder))   # 행 수 비교
world_gapminder.head()

⚠️ 행 수가 177 → 178로 늘어난 것에 주의
- gapminder 데이터셋 자체의 알려진 오류: 남한(`Korea, Rep.`)과 북한(`Korea, Dem. Rep.`)이 둘 다 `iso_alpha="KOR"`로 잘못 기록되어 있음
- `world`의 `KOR`(대한민국) 1개 행이 gapminder의 두 행과 매칭되면서 조인 결과가 1행 늘어남 → `left` 조인이라도 조인 키 중복이 있으면 행이 늘어날 수 있음을 보여주는 사례

In [ ]:
# 매칭이 안 된 국가 수 (gapminder_2007에 없는 나라, 또는 ISO 코드 체계가 다른 경우)
world_gapminder["iso_alpha"].isna().sum()

In [ ]:
# KOR로 조인된 행이 2개인 것 확인 (gapminder의 남한/북한 iso_alpha 오류)
world_gapminder[world_gapminder["id"] == "KOR"][["id", "name", "country", "pop"]]

In [ ]:
# 북한(Korea, Dem. Rep.)의 iso_alpha 오류 수정: KOR -> PRK

gapminder_2007.loc[
    gapminder_2007["country"] == "Korea, Dem. Rep.", "iso_alpha"
] = "PRK"

# 확인
gapminder_2007[gapminder_2007["country"].str.contains("Korea")][["country", "iso_alpha"]]


In [ ]:
# 다시 조인해서 문제 없이 177개 유지된것 확인
world_gapminder = world.merge(
    gapminder_2007,
    left_on="id",
    right_on="iso_alpha",
    how="left",
)

print(type(world_gapminder))
print(len(world), "->", len(world_gapminder))   # 행 수 비교
world_gapminder.head()

## Folium
- **Folium** : Leaflet.js(자바스크립트 지도 라이브러리)를 파이썬에서 감싸서 사용할 수 있게 해주는 라이브러리
  - `folium.Map()`으로 확대/축소·이동이 가능한 인터랙티브 지도를 생성
  - `folium.Choropleth()`로 GeoJSON + 데이터를 매칭해 단계 구분도(choropleth map) 시각화
  - `folium.Marker()`, `folium.GeoJson()` 등으로 마커, 팝업, 툴팁 등 다양한 레이어 추가 가능
  - Plotly의 `px.choropleth`와 달리 결과물이 실제 웹 지도(Leaflet) 위에 그려져서 줌/이동 등 실제 지도 탐색에 강점
  - 공식 사이트: https://python-visualization.github.io/folium/

- pip install folium

In [ ]:
geojson_data = json.loads(world_gapminder.to_json())  # geometry -> GeoJSON, 컬럼은 properties로 포함

m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=geojson_data,
    data=world_gapminder,
    columns=["id", "gdpPercap"],      # [키 컬럼, 값 컬럼]
    key_on="feature.properties.id",   # geojson 쪽 매칭 키
    fill_color="Blues",               # 크기(매그니튜드) -> 단일 색상 계열
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="lightgrey",       # 매칭 안 된(결측) 국가
    legend_name="1인당 GDP (2007)",
).add_to(m)

# hover 시 국가명 / GDP 값 표시 (Choropleth 자체는 tooltip을 지원하지 않아 별도 레이어 추가)
folium.GeoJson(
    geojson_data,
    style_function=lambda x: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "gdpPercap"],
        aliases=["국가", "1인당 GDP"],
        localize=True,
    ),
).add_to(m)


m.save("html/9_1.html")
m

In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import urlencode

import folium
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = (os.getenv("CARTO_API_KEY") or "").strip()

if not api_key:
    raise RuntimeError(".env에 CARTO_API_KEY를 설정하세요.")

tile_url = (
    "https://basemaps.cartocdn.com/rastertiles/light_all/"
    "{z}/{x}/{y}.png?"
    + urlencode({"key": api_key})
)

attribution = (
    '&copy; <a href="https://www.openstreetmap.org/copyright">'
    'OpenStreetMap</a> contributors '
    '&copy; <a href="https://carto.com/attributions">CARTO</a>'
)

geojson_data = json.loads(world_gapminder.to_json())

m = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles=None,
    max_zoom=20,
)

# 배경지도도 체크박스로 켜고 끄기
folium.TileLayer(
    tiles=tile_url,
    attr=attribution,
    name="CartoDB 배경지도",
    overlay=True,
    control=True,
    show=True,
    max_zoom=20,
).add_to(m)

# Choropleth는 Map에 직접 추가
choropleth = folium.Choropleth(
    geo_data=geojson_data,
    data=world_gapminder,
    columns=["id", "gdpPercap"],
    key_on="feature.properties.id",
    name="1인당 GDP (2007)",
    overlay=True,
    control=True,
    show=True,
    fill_color="Blues",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="lightgrey",
    legend_name="1인당 GDP (2007)",
).add_to(m)

# 별도 레이어 없이 단계구분도에 툴팁 연결
folium.GeoJsonTooltip(
    fields=["name", "gdpPercap"],
    aliases=["국가", "1인당 GDP"],
    localize=True,
).add_to(choropleth.geojson)

folium.LayerControl(
    position="topright",
    collapsed=False,
).add_to(m)

# GDP 레이어를 꺼도 색상 범례는 유지됩니다.
Path("html").mkdir(parents=True, exist_ok=True)
m.save("html/9_1_1.html")

m

### 4. world-centroids.geojson ↔ gapminder_2007 조인
- `world-centroids.geojson`: 국가별 대표 좌표(중심점, Point)를 담은 GeoJSON (`id`(ISO-3), `name`, `geometry`)
- `inner` 조인 → 양쪽에 모두 존재하는 국가(교집합)만 남음, `left` 조인과 달리 매칭 안 되는 행은 결측으로 남기지 않고 아예 제거

In [ ]:
world_centroids = gpd.read_file("data/world-centroids.geojson")

print(world_centroids.shape)
world_centroids.head()

In [ ]:
# world_centroids(중심점 좌표)와 gapminder_2007을 inner 조인
centroids_gapminder = world_centroids.merge(
    gapminder_2007,
    left_on="id",
    right_on="iso_alpha",
    how="inner",
)

print(len(world_centroids), "->", len(centroids_gapminder))   # 교집합만 남아 행 수가 줄어듦
centroids_gapminder.head()

### 5. Folium으로 gdpPercap · lifeExp 시각화

In [ ]:
geojson_data = json.loads(world_gapminder.to_json())  # geometry -> GeoJSON, 컬럼은 properties로 포함

m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=geojson_data,
    data=world_gapminder,
    columns=["id", "gdpPercap"],      # [키 컬럼, 값 컬럼]
    key_on="feature.properties.id",   # geojson 쪽 매칭 키
    fill_color="Blues",               # 크기(매그니튜드) -> 단일 색상 계열
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="lightgrey",       # 매칭 안 된(결측) 국가
    legend_name="1인당 GDP (2007)",
).add_to(m)

# hover 시 국가명 / GDP 값 표시 (Choropleth 자체는 tooltip을 지원하지 않아 별도 레이어 추가)
folium.GeoJson(
    geojson_data,
    style_function=lambda x: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "gdpPercap"],
        aliases=["국가", "1인당 GDP"],
        localize=True,
    ),
).add_to(m)

# centroids_gapminder의 lifeExp를 point layer로 추가
# - 배경 choropleth(gdpPercap)는 Blues 계열이므로, 구분되도록 다른 색상 계열(YlOrRd) 사용
# - lifeExp가 클수록 점 크기(반지름)와 색상(진하기)이 함께 커지도록 매핑
import branca.colormap as cm

lifeExp_min, lifeExp_max = centroids_gapminder["lifeExp"].min(), centroids_gapminder["lifeExp"].max()
lifeExp_colormap = cm.linear.YlOrRd_09.scale(lifeExp_min, lifeExp_max)
lifeExp_colormap.caption = "기대수명 (lifeExp)"

MIN_RADIUS, MAX_RADIUS = 2, 10

def lifeExp_to_radius(value):
    ratio = (value - lifeExp_min) / (lifeExp_max - lifeExp_min)
    return MIN_RADIUS + ratio * (MAX_RADIUS - MIN_RADIUS)

for _, row in centroids_gapminder.iterrows():
    lon, lat = row.geometry.x, row.geometry.y
    folium.CircleMarker(
        location=[lat, lon],
        radius=lifeExp_to_radius(row["lifeExp"]),
        color=lifeExp_colormap(row["lifeExp"]),
        weight=1,
        fill=True,
        fill_color=lifeExp_colormap(row["lifeExp"]),
        fill_opacity=0.70,
        tooltip=f"{row['name']}<br>기대수명: {row['lifeExp']:.1f}세",
    ).add_to(m)

lifeExp_colormap.add_to(m)

m.save("html/9_2.html")

m